# 📘 Documentación Técnica y Pruebas del Código: Hundir la Flota

Este cuaderno interactivo detalla las decisiones de ingeniería de software, la vectorización con **NumPy** y la estructura **Orientada a Objetos** implementada en el proyecto, además de ejecutar las pruebas funcionales solicitadas.

## 1. Creación del Tablero con NumPy (`crear_tablero`)

Para representar el mar utilizamos `np.full((tamaño, tamaño), '_', dtype='<U1')`. 
* **Justificación técnica:** A diferencia de las listas nativas de Python, un array de NumPy almacena los datos en un bloque contiguo de memoria RAM. Esto permite consultar o modificar subregiones completas (mediante *slicing*) en tiempo constante $O(1)$.

In [ ]:
import numpy as np
import variables as var
import funciones as fn
import utils

# Creamos un tablero por defecto de 10x10
tablero_demo = fn.crear_tablero(10)
utils.mostrar_tablero_individual(tablero_demo, "TABLERO INICIAL 10x10")

## 2. Posicionamiento Manual de Barcos (`colocar_barco` - Requisito 6)

A continuación verificamos la función `colocar_barco(barco, tablero)` posicionando explícitamente los dos barcos indicados en las especificaciones:
* **Barco 1 (eslora 2):** `[(0, 1), (1, 1)]`
* **Barco 2 (eslora 4):** `[(1, 3), (1, 4), (1, 5), (1, 6)]`

In [ ]:
barco_2_casillas = [(0, 1), (1, 1)]
barco_4_casillas = [(1, 3), (1, 4), (1, 5), (1, 6)]

fn.colocar_barco(barco_2_casillas, tablero_demo)
fn.colocar_barco(barco_4_casillas, tablero_demo)

utils.mostrar_tablero_individual(tablero_demo, "PRUEBA DE COLOCACIÓN MANUAL")

## 3. Prueba de la Función `disparar(casilla, tablero)` (Requisito 8)

Ponemos a prueba la función `disparar` atacando el barco de 2 casillas ubicado en `(0, 1)` y `(1, 1)`, así como una casilla de agua en `(4, 4)`:
* Al impactar sobre `'O'`, el carácter se transforma en `'X'` y se informa de **Tocado**.
* Al impactar sobre `'_'`, el carácter se transforma en `'A'` y se informa de **Agua**.

In [ ]:
print("--- Disparo 1: Coordenada (0, 1) ---")
fn.disparar((0, 1), tablero_demo)

print("\n--- Disparo 2: Coordenada (1, 1) ---")
fn.disparar((1, 1), tablero_demo)

print("\n--- Disparo 3: Coordenada (4, 4) ---")
fn.disparar((4, 4), tablero_demo)

utils.mostrar_tablero_individual(tablero_demo, "TABLERO TRAS LOS DISPAROS")

## 4. Algoritmo de Halo de Seguridad y Generación de Flota (`colocar_barcos` - Requisito 7)

Para garantizar que ningún barco se superponga ni quede pegado a otro (manteniendo al menos **1 casilla de separación horizontal y vertical**), la función `validar_posicion_barco` calcula los valores mínimos y máximos de fila y columna del barco candidato y amplía ese rango en 1 casilla por cada lado:

```python
fila_min = max(0, min(filas) - 1)
fila_max = min(tamaño, max(filas) + 2)
col_min = max(0, min(columnas) - 1)
col_max = min(tamaño, max(columnas) + 2)
submatriz_entorno = tablero[fila_min:fila_max, col_min:col_max]
```

Si `np.any(submatriz_entorno == 'O')` devuelve `True`, significa que existe otro barco en la propia ubicación o en su perímetro inmediato de 1 casilla, por lo que se descarta y se genera otra posición.

In [ ]:
tablero_flota_aleatoria = fn.crear_tablero(10)
fn.colocar_barcos(tablero_flota_aleatoria)

utils.mostrar_tablero_individual(tablero_flota_aleatoria, "FLOTA COMPLETA ALEATORIA (6 BARCOS CON MARGEN)")
print(f"Total de casillas ocupadas por barcos ('O'): {np.sum(tablero_flota_aleatoria == 'O')} (Esperado: 16)")

## 5. Diseño Orientado a Objetos y Selección de Disparos de la IA

En `utils.py` se estructuran tres clases fundamentales:
1. **`Tablero`:** Encapsula `matriz_flota` y `matriz_disparos`.
2. **`Jugador`:** Representa tanto al jugador humano como a la máquina. La máquina emplea `np.argwhere(self.tablero.matriz_disparos == '_')` para obtener instantáneamente el listado de coordenadas vírgenes, evitando repetir disparos pasados.
3. **`JuegoHundirLaFlota`:** Gestiona el bucle de turnos, la repetición de turno tras un acierto y la limpieza de la consola.